# Partial Cross Entropy Loss for Weakly Supervised Segmentation

## Problem Statement
In many remote sensing applications, obtaining full pixel-level annotations is expensive and time-consuming. Point annotations provide a more practical alternative, but standard deep learning losses require complete segmentation masks.

## Solution Overview
This notebook implements:
1. **Partial Cross Entropy Loss** - A modified loss that only computes loss on labeled pixels
2. **Point Label Simulation** - Converting full masks to sparse point annotations
3. **Experiments** - Exploring factors affecting performance

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
import random
from tqdm import tqdm
import copy

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'torch'

## 1. Partial Cross Entropy Loss Implementation

The key idea is to only compute the cross-entropy loss on pixels that have labels (i.e., the point annotations), while ignoring unlabeled pixels.

In [ ]:
class PartialCrossEntropyLoss(nn.Module):
    """
    Partial Cross Entropy Loss for weakly supervised segmentation.
    
    Only computes loss on labeled pixels (where label_mask is True).
    Unlabeled pixels are ignored in the loss computation.
    
    Args:
        ignore_index: Index to ignore in the target (default: -1)
        reduction: 'mean', 'sum', or 'none'
        label_smoothing: Label smoothing parameter
    """
    def __init__(self, ignore_index=-1, reduction='mean', label_smoothing=0.0):
        super(PartialCrossEntropyLoss, self).__init__()
        self.ignore_index = ignore_index
        self.reduction = reduction
        self.label_smoothing = label_smoothing
    
    def forward(self, pred, target, label_mask=None):
        """
        Args:
            pred: (B, C, H, W) - raw logits from the model
            target: (B, H, W) - ground truth labels (with -1 for unlabeled)
            label_mask: (B, H, W) - boolean mask indicating labeled pixels
                     If None, derived from target != ignore_index
        """
        B, C, H, W = pred.shape
        
        # Derive label_mask from target if not provided
        if label_mask is None:
            label_mask = (target != self.ignore_index)
        
        # Flatten tensors
        pred_flat = pred.permute(0, 2, 3, 1).contiguous().view(-1, C)  # (B*H*W, C)
        target_flat = target.view(-1)  # (B*H*W,)
        label_mask_flat = label_mask.view(-1)  # (B*H*W,)
        
        # Only consider labeled pixels
        labeled_indices = label_mask_flat.nonzero(as_tuple=True)[0]
        
        if len(labeled_indices) == 0:
            # No labeled pixels in this batch
            return torch.tensor(0.0, device=pred.device, requires_grad=True)
        
        pred_labeled = pred_flat[labeled_indices]
        target_labeled = target_flat[labeled_indices]
        
        # Apply label smoothing if specified
        if self.label_smoothing > 0:
            # Smooth labels: (1 - eps) * one_hot + eps / C
            n_classes = pred.size(1)
            one_hot = F.one_hot(target_labeled, num_classes=n_classes).float()
            smoothed_labels = (1 - self.label_smoothing) * one_hot + 
                              self.label_smoothing / n_classes
            log_prob = F.log_softmax(pred_labeled, dim=-1)
            loss = -(smoothed_labels * log_prob).sum(dim=-1)
        else:
            # Standard cross-entropy
            loss = F.cross_entropy(pred_labeled, target_labeled, reduction='none')
        
        # Apply reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


# Test the loss function
def test_partial_ce_loss():
    # Create dummy predictions and targets
    B, C, H, W = 2, 4, 8, 8
    pred = torch.randn(B, C, H, W)
    
    # Create target with partial labels
    target = torch.full((B, H, W), -1, dtype=torch.long)  # All unlabeled
    target[0, 2:4, 2:4] = 0  # Label some pixels
    target[1, 5:7, 5:7] = 1
    
    # Create label mask
    label_mask = (target != -1)
    
    loss_fn = PartialCrossEntropyLoss()
    loss = loss_fn(pred, target, label_mask)
    
    print(f"Partial CE Loss: {loss.item():.4f}")
    print(f"Labeled pixels: {label_mask.sum().item()}/{label_mask.numel()}")
    return loss

test_partial_ce_loss()

## 2. Point Label Generation

Simulate point annotations by sampling pixels from full segmentation masks.

In [ ]:
def generate_point_labels(mask, num_points_per_class=5, strategy='random'):
    """
    Generate point annotations from a full segmentation mask.
    
    Args:
        mask: (H, W) - full segmentation mask
        num_points_per_class: Number of points to sample per class
        strategy: 'random', 'grid', 'entropy', 'boundary'
    
    Returns:
        point_mask: (H, W) - mask with -1 for unlabeled, class label for labeled
        label_positions: list of (r, c) tuples of labeled pixel positions
    """
    H, W = mask.shape
    point_mask = torch.full((H, W), -1, dtype=torch.long)
    unique_classes = torch.unique(mask)
    unique_classes = unique_classes[unique_classes >= 0]  # Ignore background if needed
    
    label_positions = []
    
    for cls in unique_classes:
        cls_mask = (mask == cls)
        cls_pixels = torch.nonzero(cls_mask, as_tuple=False)
        
        if len(cls_pixels) == 0:
            continue
        
        if strategy == 'random':
            # Random sampling
            num_samples = min(num_points_per_class, len(cls_pixels))
            indices = torch.randperm(len(cls_pixels))[:num_samples]
            selected = cls_pixels[indices]
            
        elif strategy == 'grid':
            # Grid-based sampling
            num_samples = min(num_points_per_class, len(cls_pixels))
            grid_size = int(np.sqrt(num_samples)) + 1
            selected = []
            for i in range(grid_size):
                for j in range(grid_size):
                    if len(selected) >= num_samples:
                        break
                    # Compute grid position
                    r = (i * cls_pixels[:, 0].max().item() // grid_size + 
                         cls_pixels[:, 0].min().item()) // 2
                    c = (j * cls_pixels[:, 1].max().item() // grid_size + 
                         cls_pixels[:, 1].min().item()) // 2
                    # Find nearest pixel of this class
                    distances = torch.sqrt((cls_pixels[:, 0] - r)**2 + 
                                          (cls_pixels[:, 1] - c)**2)
                    nearest = cls_pixels[distances.argmin()]
                    selected.append(nearest)
            selected = torch.stack(selected) if selected else cls_pixels[:1]
            
        elif strategy == 'boundary':
            # Sample near boundaries (more informative)
            from scipy.ndimage import binary_erosion
            cls_mask_np = cls_mask.cpu().numpy()
            eroded = binary_erosion(cls_mask_np)
            boundary = cls_mask_np & ~eroded
            boundary_pixels = torch.nonzero(torch.from_numpy(boundary), as_tuple=False)
            
            if len(boundary_pixels) > 0:
                num_samples = min(num_points_per_class, len(boundary_pixels))
                indices = torch.randperm(len(boundary_pixels))[:num_samples]
                selected = boundary_pixels[indices]
            else:
                selected = cls_pixels[:1]
        
        else:
            selected = cls_pixels[:min(num_points_per_class, len(cls_pixels))]
        
        for r, c in selected:
            point_mask[r, c] = cls.item()
            label_positions.append((r.item(), c.item()))
    
    return point_mask, label_positions


# Visualize point labels
def visualize_point_labels(image, full_mask, point_mask, title="Point Annotations"):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Full mask
    axes[1].imshow(full_mask, cmap='tab20')
    axes[1].set_title('Full Segmentation Mask')
    axes[1].axis('off')
    
    # Point annotations
    axes[2].imshow(image)
    labeled_mask = (point_mask != -1)
    labeled_pixels = torch.nonzero(labeled_mask, as_tuple=False)
    
    colors = plt.cm.tab20(torch.linspace(0, 1, 20))
    for r, c in labeled_pixels:
        cls = point_mask[r, c].item()
        axes[2].scatter(c, r, c=[colors[cls]], s=100, marker='o', edgecolors='white', linewidths=1)
    
    axes[2].set_title(f'{title}\n({len(labeled_pixels)} points)')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()


# Test point label generation
def test_point_generation():
    # Create a synthetic mask
    H, W = 128, 128
    mask = torch.zeros(H, W, dtype=torch.long)
    mask[:64, :64] = 0
    mask[:64, 64:] = 1
    mask[64:, :64] = 2
    mask[64:, 64:] = 3
    
    # Generate point labels
    point_mask, positions = generate_point_labels(mask, num_points_per_class=10, strategy='random')
    
    # Create a dummy image
    image = torch.randn(H, W, 3).clip(0, 1)
    
    visualize_point_labels(image, mask, point_mask)
    
    return point_mask

print("Testing point label generation...")
test_point_generation()

## 3. Segmentation Model

A lightweight U-Net style architecture for segmentation.

In [ ]:
class DoubleConv(nn.Module):
    """(Conv2D -> BN -> ReLU) x 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class UNetLite(nn.Module):
    """Lightweight U-Net for segmentation."""
    def __init__(self, in_channels=3, num_classes=5):
        super().__init__()
        
        # Encoder
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)
        
        # Decoder
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        
        # Final convolution
        self.final = nn.Conv2d(32, num_classes, 1)
        
        self.pool = nn.MaxPool2d(2)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool(e2)
        
        e3 = self.enc3(p2)
        p3 = self.pool(e3)
        
        e4 = self.enc4(p3)
        
        # Decoder
        d3 = self.up3(e4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        return self.final(d1)


class DeepLabV3Lite(nn.Module):
    """Simplified DeepLabV3 with ASPP."""
    def __init__(self, in_channels=3, num_classes=5):
        super().__init__()
        
        # Backbone (simplified ResNet)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        
        self.conv2 = self._make_layer(64, 128, 2)
        self.conv3 = self._make_layer(128, 256, 2)
        self.conv4 = self._make_layer(256, 512, 2)
        
        # ASPP
        self.aspp = ASPP(512, 256)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Conv2d(256, num_classes, 1)
        )
    
    def _make_layer(self, in_channels, out_channels, blocks):
        layers = []
        layers.append(nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ))
        for _ in range(blocks - 1):
            layers.append(nn.Sequential(
                nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        input_size = x.shape[-2:]
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.aspp(x)
        x = self.decoder(x)
        return F.interpolate(x, size=input_size, mode='bilinear', align_corners=False)


class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        rates = [6, 12, 18]
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ) for r in rates
        ])
        
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.project = nn.Sequential(
            nn.Conv2d(out_channels * (len(rates) + 2), out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )
    
    def forward(self, x):
        size = x.shape[-2:]
        res = [self.conv1(x)]
        res.extend([conv(x) for conv in self.convs])
        res.append(F.interpolate(self.pool(x), size=size, mode='bilinear', align_corners=False))
        res = torch.cat(res, dim=1)
        return self.project(res)


# Test model
def test_model():
    model = UNetLite(in_channels=3, num_classes=5).to(device)
    x = torch.randn(2, 3, 128, 128).to(device)
    out = model(x)
    print(f"Model output shape: {out.shape}")
    print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model

print("\nTesting model...")
model = test_model()

## 4. Dataset and Data Loading

Using a synthetic remote sensing dataset for demonstration.

In [ ]:
class SyntheticRemoteSensingDataset(Dataset):
    """
    Synthetic remote sensing dataset for segmentation.
    Generates synthetic images with various land cover types.
    """
    def __init__(self, num_samples=500, img_size=256, num_classes=5, transform=None):
        self.num_samples = num_samples
        self.img_size = img_size
        self.num_classes = num_classes
        self.transform = transform
        
        # Define color patterns for different classes
        self.class_colors = {
            0: [34, 139, 34],    # Forest - green
            1: [210, 180, 140],  # Urban - tan
            2: [65, 105, 225],   # Water - blue
            3: [240, 230, 140],  # Agricultural - yellow
            4: [139, 69, 19],    # Bare soil - brown
        }
    
    def __len__(self):
        return self.num_samples
    
    def generate_sample(self, idx):
        # Generate random mask using Voronoi-like regions
        mask = torch.zeros(self.img_size, self.img_size, dtype=torch.long)
        
        # Generate random seed points for each class
        num_seeds_per_class = random.randint(2, 5)
        seeds = []
        for cls in range(self.num_classes):
            for _ in range(num_seeds_per_class):
                seeds.append({
                    'class': cls,
                    'x': random.randint(0, self.img_size - 1),
                    'y': random.randint(0, self.img_size - 1)
                })
        
        # Assign each pixel to nearest seed (Voronoi)
        from scipy.spatial import cKDTree
        seed_coords = np.array([[s['x'], s['y']] for s in seeds])
        seed_classes = np.array([s['class'] for s in seeds])
        
        y_coords, x_coords = np.meshgrid(np.arange(self.img_size), np.arange(self.img_size))
        pixels = np.column_stack([x_coords.ravel(), y_coords.ravel()])
        
        tree = cKDTree(seed_coords)
        _, indices = tree.query(pixels)
        mask = torch.from_numpy(seed_classes[indices].reshape(self.img_size, self.img_size)).long()
        
        # Add some noise/smoothness
        from scipy.ndimage import binary_dilation
        for _ in range(2):
            new_mask = mask.clone()
            for i in range(self.img_size):
                for j in range(self.img_size):
                    if random.random() < 0.1:
                        # Look at neighbors
                        neighbors = []
                        for di in [-1, 0, 1]:
                            for dj in [-1, 0, 1]:
                                ni, nj = i + di, j + dj
                                if 0 <= ni < self.img_size and 0 <= nj < self.img_size:
                                    neighbors.append(mask[ni, nj].item())
                        if neighbors:
                            new_mask[i, j] = max(set(neighbors), key=neighbors.count)
            mask = new_mask
        
        # Generate image from mask
        image = torch.zeros(self.img_size, self.img_size, 3)
        for cls, color in self.class_colors.items():
            for c in range(3):
                # Add variation
                base_color = color[c] / 255.0
                noise = torch.randn(self.img_size, self.img_size) * 0.05
                image[:, :, c] += (mask == cls).float() * (base_color + noise)
        
        image = image.clamp(0, 1)
        
        return image, mask
    
    def __getitem__(self, idx):
        image, mask = self.generate_sample(idx)
        
        if self.transform:
            image = self.transform(image)
        
        return image, mask


class PointLabelDataset(Dataset):
    """
    Wrapper that converts full masks to point labels.
    """
    def __init__(self, base_dataset, num_points_per_class=5, point_strategy='random'):
        self.base_dataset = base_dataset
        self.num_points_per_class = num_points_per_class
        self.point_strategy = point_strategy
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, full_mask = self.base_dataset[idx]
        
        # Generate point labels from full mask
        point_mask, _ = generate_point_labels(
            full_mask.squeeze(0) if full_mask.dim() > 2 else full_mask,
            num_points_per_class=self.num_points_per_class,
            strategy=self.point_strategy
        )
        
        return image, point_mask, full_mask


# Create datasets
def create_datasets(num_train=400, num_val=100, img_size=256):
    transform = transforms.Compose([
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    train_dataset = SyntheticRemoteSensingDataset(
        num_samples=num_train, img_size=img_size, transform=transform
    )
    val_dataset = SyntheticRemoteSensingDataset(
        num_samples=num_val, img_size=img_size, transform=transform
    )
    
    return train_dataset, val_dataset


# Visualize dataset samples
def visualize_dataset_samples(dataset, num_samples=4):
    fig, axes = plt.subplots(2, num_samples, figsize=(4*num_samples, 8))
    
    for i in range(num_samples):
        image, mask = dataset[i]
        
        # Denormalize for visualization
        image_vis = image * 0.5 + 0.5
        image_vis = image_vis.permute(1, 2, 0).clamp(0, 1)
        
        axes[0, i].imshow(image_vis)
        axes[0, i].set_title(f'Image {i+1}')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(mask.squeeze(), cmap='tab20')
        axes[1, i].set_title(f'Mask {i+1}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()


print("Creating datasets...")
train_dataset, val_dataset = create_datasets()
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print("\nVisualizing samples:")
visualize_dataset_samples(train_dataset)

## 5. Training Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, use_point_labels=True):
    """
    Train for one epoch.
    
    Args:
        use_point_labels: If True, use point labels (3rd element from dataset)
    """
    model.train()
    total_loss = 0.0
    total_pixels = 0
    labeled_pixels_count = 0
    
    for batch in dataloader:
        if use_point_labels:
            images, point_masks, full_masks = batch
        else:
            images, full_masks = batch
            point_masks = None
        
        images = images.to(device)
        masks = point_masks if use_point_labels else full_masks
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        
        # Compute loss
        if use_point_labels:
            # Partial CE loss - only labeled pixels contribute
            label_mask = (masks != -1)
            loss = criterion(outputs, masks, label_mask)
            labeled_pixels_count += label_mask.sum().item()
        else:
            # Standard CE loss
            loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        total_pixels += images.size(0) * images.size(2) * images.size(3)
    
    avg_loss = total_loss / len(dataloader)
    labeled_ratio = labeled_pixels_count / total_pixels if use_point_labels else 1.0
    
    return avg_loss, labeled_ratio


@torch.no_grad()
def validate(model, dataloader, criterion, device, num_classes):
    """
    Validate the model.
    
    Returns metrics computed on full masks (for fair evaluation).
    """
    model.eval()
    total_loss = 0.0
    
    # For IoU calculation
    intersection = torch.zeros(num_classes, device=device)
    union = torch.zeros(num_classes, device=device)
    correct = 0
    total = 0
    
    for batch in dataloader:
        if len(batch) == 3:
            images, _, full_masks = batch
        else:
            images, full_masks = batch
        
        images = images.to(device)
        full_masks = full_masks.to(device)
        
        # Forward pass
        outputs = model(images)
        
        # Compute loss on full mask
        loss = criterion(outputs, full_masks)
        total_loss += loss.item() * images.size(0)
        
        # Compute predictions
        preds = outputs.argmax(dim=1)
        
        # Compute pixel accuracy
        correct += (preds == full_masks).sum().item()
        total += full_masks.numel()
        
        # Compute IoU per class
        for cls in range(num_classes):
            pred_cls = (preds == cls)
            mask_cls = (full_masks == cls)
            intersection[cls] += (pred_cls & mask_cls).sum().item()
            union[cls] += (pred_cls | mask_cls).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    pixel_accuracy = correct / total
    
    # Mean IoU
    iou_per_class = intersection / (union + 1e-8)
    miou = iou_per_class.mean().item()
    
    return {
        'loss': avg_loss,
        'pixel_accuracy': pixel_accuracy,
        'miou': miou,
        'iou_per_class': iou_per_class.cpu().numpy()
    }


def train_model(model, train_loader, val_loader, num_epochs, lr, device, 
                use_point_labels=True, num_classes=5):
    """
    Train a model with partial CE loss.
    """
    if use_point_labels:
        criterion = PartialCrossEntropyLoss(ignore_index=-1)
    else:
        criterion = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_acc': []}
    best_miou = 0.0
    best_model = None
    
    for epoch in range(num_epochs):
        train_loss, labeled_ratio = train_epoch(
            model, train_loader, criterion, optimizer, device, use_point_labels
        )
        
        val_metrics = validate(model, val_loader, nn.CrossEntropyLoss(), device, num_classes)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_miou'].append(val_metrics['miou'])
        history['val_acc'].append(val_metrics['pixel_accuracy'])
        
        scheduler.step()
        
        # Save best model
        if val_metrics['miou'] > best_miou:
            best_miou = val_metrics['miou']
            best_model = copy.deepcopy(model.state_dict())
        
        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"Train Loss: {train_loss:.4f}, "
              f"Val Loss: {val_metrics['loss']:.4f}, "
              f"Val mIoU: {val_metrics['miou']:.4f}, "
              f"Val Acc: {val_metrics['pixel_accuracy']:.4f}", end="")
        if use_point_labels:
            print(f", Labeled: {labeled_ratio*100:.2f}%")
        else:
            print()
    
    # Load best model
    if best_model is not None:
        model.load_state_dict(best_model)
    
    return history, best_miou

## 6. Experiments

### Experiment 1: Effect of Point Annotation Density

In [ ]:
def experiment_point_density():
    """
    Experiment: How does the number of point annotations per class affect performance?
    
    Hypothesis: More point annotations should improve performance, but with diminishing returns.
    """
    print("="*60)
    print("EXPERIMENT 1: Effect of Point Annotation Density")
    print("="*60)
    print("\nHypothesis: More point annotations per class will improve performance,")
    print("but with diminishing returns.")
    print("\nIndependent Variable: Number of points per class (1, 3, 5, 10, 20)")
    print("Dependent Variables: mIoU, Pixel Accuracy")
    print("\n" + "="*60)
    
    # Experiment parameters
    points_per_class_list = [1, 3, 5, 10, 20]
    num_epochs = 20
    batch_size = 16
    lr = 0.001
    img_size = 128
    num_classes = 5
    
    results = []
    
    for num_points in points_per_class_list:
        print(f"\n--- Training with {num_points} points per class ---")
        
        # Create datasets
        train_base, val_base = create_datasets(num_train=200, num_val=50, img_size=img_size)
        
        train_dataset = PointLabelDataset(
            train_base, num_points_per_class=num_points, point_strategy='random'
        )
        val_dataset = PointLabelDataset(
            val_base, num_points_per_class=num_points, point_strategy='random'
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Create model
        model = UNetLite(in_channels=3, num_classes=num_classes).to(device)
        
        # Train
        history, best_miou = train_model(
            model, train_loader, val_loader, 
            num_epochs=num_epochs, lr=lr, device=device,
            use_point_labels=True, num_classes=num_classes
        )
        
        # Final evaluation
        val_metrics = validate(model, val_loader, nn.CrossEntropyLoss(), device, num_classes)
        
        results.append({
            'num_points': num_points,
            'miou': val_metrics['miou'],
            'accuracy': val_metrics['pixel_accuracy'],
            'history': history
        })
        
        print(f"\nFinal mIoU: {val_metrics['miou']:.4f}, Accuracy: {val_metrics['pixel_accuracy']:.4f}")
    
    return results


# Run Experiment 1
# Uncomment to run:
# results_density = experiment_point_density()

### Experiment 2: Point Annotation Strategy

In [ ]:
def experiment_point_strategy():
    """
    Experiment: How does the point sampling strategy affect performance?
    
    Hypothesis: Strategic sampling (e.g., boundary-focused) will outperform random sampling
    for the same number of points.
    """
    print("="*60)
    print("EXPERIMENT 2: Effect of Point Sampling Strategy")
    print("="*60)
    print("\nHypothesis: Strategic sampling strategies will outperform random sampling")
    print("for the same number of points.")
    print("\nIndependent Variable: Sampling strategy (random, grid, boundary)")
    print("Dependent Variables: mIoU, Pixel Accuracy")
    print("\n" + "="*60)
    
    strategies = ['random', 'grid']  # boundary requires scipy
    num_points_per_class = 5
    num_epochs = 20
    batch_size = 16
    lr = 0.001
    img_size = 128
    num_classes = 5
    
    results = []
    
    for strategy in strategies:
        print(f"\n--- Training with {strategy} sampling strategy ---")
        
        # Create datasets
        train_base, val_base = create_datasets(num_train=200, num_val=50, img_size=img_size)
        
        train_dataset = PointLabelDataset(
            train_base, num_points_per_class=num_points_per_class, point_strategy=strategy
        )
        val_dataset = PointLabelDataset(
            val_base, num_points_per_class=num_points_per_class, point_strategy=strategy
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Create model
        model = UNetLite(in_channels=3, num_classes=num_classes).to(device)
        
        # Train
        history, best_miou = train_model(
            model, train_loader, val_loader, 
            num_epochs=num_epochs, lr=lr, device=device,
            use_point_labels=True, num_classes=num_classes
        )
        
        # Final evaluation
        val_metrics = validate(model, val_loader, nn.CrossEntropyLoss(), device, num_classes)
        
        results.append({
            'strategy': strategy,
            'miou': val_metrics['miou'],
            'accuracy': val_metrics['pixel_accuracy'],
            'history': history
        })
        
        print(f"\nFinal mIoU: {val_metrics['miou']:.4f}, Accuracy: {val_metrics['pixel_accuracy']:.4f}")
    
    return results


# Run Experiment 2
# Uncomment to run:
# results_strategy = experiment_point_strategy()

### Experiment 3: Point vs. Full Supervision Comparison

In [ ]:
def experiment_supervision_comparison():
    """
    Experiment: Compare partial CE loss (point supervision) vs. full supervision.
    
    Hypothesis: Point supervision will achieve comparable results to full supervision
    when using sufficient points and appropriate loss function.
    """
    print("="*60)
    print("EXPERIMENT 3: Point vs. Full Supervision Comparison")
    print("="*60)
    print("\nHypothesis: With sufficient labeled points, partial CE loss will approach")
    print("the performance of full supervision.")
    print("\nConditions: Point supervision (5 points/class) vs. Full supervision")
    print("Dependent Variables: mIoU, Pixel Accuracy, Training Speed")
    print("\n" + "="*60)
    
    num_epochs = 30
    batch_size = 16
    lr = 0.001
    img_size = 128
    num_classes = 5
    num_points = 5
    
    results = {}
    
    # Create base datasets
    train_base, val_base = create_datasets(num_train=200, num_val=50, img_size=img_size)
    
    # ===== Point Supervision =====
    print("\n--- Point Supervision (5 points per class) ---")
    train_point = PointLabelDataset(train_base, num_points_per_class=num_points)
    val_point = PointLabelDataset(val_base, num_points_per_class=num_points)
    
    train_loader_p = DataLoader(train_point, batch_size=batch_size, shuffle=True)
    val_loader_p = DataLoader(val_point, batch_size=batch_size, shuffle=False)
    
    model_point = UNetLite(in_channels=3, num_classes=num_classes).to(device)
    
    history_point, best_miou_point = train_model(
        model_point, train_loader_p, val_loader_p,
        num_epochs=num_epochs, lr=lr, device=device,
        use_point_labels=True, num_classes=num_classes
    )
    
    metrics_point = validate(model_point, val_loader_p, nn.CrossEntropyLoss(), device, num_classes)
    
    results['point'] = {
        'miou': metrics_point['miou'],
        'accuracy': metrics_point['pixel_accuracy'],
        'history': history_point
    }
    
    print(f"Point Supervision - mIoU: {metrics_point['miou']:.4f}, Acc: {metrics_point['pixel_accuracy']:.4f}")
    
    # ===== Full Supervision =====
    print("\n--- Full Supervision ---")
    train_loader_f = DataLoader(train_base, batch_size=batch_size, shuffle=True)
    val_loader_f = DataLoader(val_base, batch_size=batch_size, shuffle=False)
    
    model_full = UNetLite(in_channels=3, num_classes=num_classes).to(device)
    
    history_full, best_miou_full = train_model(
        model_full, train_loader_f, val_loader_f,
        num_epochs=num_epochs, lr=lr, device=device,
        use_point_labels=False, num_classes=num_classes
    )
    
    metrics_full = validate(model_full, val_loader_f, nn.CrossEntropyLoss(), device, num_classes)
    
    results['full'] = {
        'miou': metrics_full['miou'],
        'accuracy': metrics_full['pixel_accuracy'],
        'history': history_full
    }
    
    print(f"Full Supervision - mIoU: {metrics_full['miou']:.4f}, Acc: {metrics_full['pixel_accuracy']:.4f}")
    
    # Summary
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Point Supervision mIoU: {results['point']['miou']:.4f}")
    print(f"Full Supervision mIoU:  {results['full']['miou']:.4f}")
    print(f"\nPerformance Gap: {(results['full']['miou'] - results['point']['miou']):.4f}")
    print(f"Relative Performance: {(results['point']['miou'] / results['full']['miou'] * 100):.1f}%")
    print("="*60)
    
    return results


# Run Experiment 3
# Uncomment to run:
# results_comparison = experiment_supervision_comparison()

## 7. Visualization Functions

In [ ]:
def plot_training_history(histories, labels, metric='val_miou'):
    """
    Plot training history comparing different runs.
    """
    plt.figure(figsize=(10, 6))
    
    for history, label in zip(histories, labels):
        epochs = range(1, len(history[metric]) + 1)
        plt.plot(epochs, history[metric], marker='o', label=label, linewidth=2)
    
    plt.xlabel('Epoch')
    plt.ylabel('Validation mIoU' if metric == 'val_miou' else metric)
    plt.title('Training Progress Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_experiment_results(results, x_key, title):
    """
    Plot experiment results.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x_values = [r[x_key] for r in results]
    mious = [r['miou'] for r in results]
    accuracies = [r['accuracy'] for r in results]
    
    axes[0].plot(x_values, mious, marker='o', linewidth=2, markersize=8)
    axes[0].set_xlabel(x_key.replace('_', ' ').title())
    axes[0].set_ylabel('Mean IoU')
    axes[0].set_title('Mean IoU vs ' + x_key.replace('_', ' ').title())
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(x_values, accuracies, marker='s', linewidth=2, markersize=8, color='orange')
    axes[1].set_xlabel(x_key.replace('_', ' ').title())
    axes[1].set_ylabel('Pixel Accuracy')
    axes[1].set_title('Pixel Accuracy vs ' + x_key.replace('_', ' ').title())
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def visualize_predictions(model, dataset, num_samples=4):
    """
    Visualize model predictions on validation samples.
    """
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    indices = random.sample(range(len(dataset)), num_samples)
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, point_mask, full_mask = dataset[idx]
            
            # Get prediction
            image_input = image.unsqueeze(0).to(device)
            output = model(image_input)
            pred = output.argmax(dim=1).squeeze().cpu()
            
            # Denormalize image
            image_vis = image * 0.5 + 0.5
            image_vis = image_vis.permute(1, 2, 0).clamp(0, 1)
            
            # Plot
            axes[i, 0].imshow(image_vis)
            axes[i, 0].set_title('Image')
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(point_mask, cmap='tab20')
            # Overlay points
            labeled_mask = (point_mask != -1)
            for r in range(point_mask.shape[0]):
                for c in range(point_mask.shape[1]):
                    if labeled_mask[r, c]:
                        axes[i, 1].scatter(c, r, c='red', s=20)
            axes[i, 1].set_title('Point Labels')
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(full_mask.squeeze(), cmap='tab20')
            axes[i, 2].set_title('Ground Truth')
            axes[i, 2].axis('off')
            
            axes[i, 3].imshow(pred, cmap='tab20')
            axes[i, 3].set_title('Prediction')
            axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

## 8. Quick Demo (Small Scale Training)

Run a quick demonstration of the partial CE loss with minimal training.

In [ ]:
def run_demo():
    """
    Quick demo showing the partial CE loss in action.
    """
    print("="*60)
    print("PARTIAL CE LOSS DEMO")
    print("="*60)
    
    # Small dataset for quick demo
    img_size = 128
    num_classes = 5
    num_points = 5
    
    print("\nCreating synthetic datasets...")
    train_base, val_base = create_datasets(num_train=100, num_val=20, img_size=img_size)
    
    # Create point label dataset
    train_dataset = PointLabelDataset(train_base, num_points_per_class=num_points)
    val_dataset = PointLabelDataset(val_base, num_points_per_class=num_points)
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    print(f"Points per class: {num_points}")
    
    # Show a sample
    sample_img, sample_points, sample_full = train_dataset[0]
    labeled_ratio = (sample_points != -1).sum() / sample_points.numel()
    print(f"Labeled pixel ratio: {labeled_ratio*100:.2f}%")
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    
    # Create model
    print("\nCreating U-Net Lite model...")
    model = UNetLite(in_channels=3, num_classes=num_classes).to(device)
    
    # Train
    print("\nTraining with Partial CE Loss...")
    history, best_miou = train_model(
        model, train_loader, val_loader,
        num_epochs=10, lr=0.001, device=device,
        use_point_labels=True, num_classes=num_classes
    )
    
    # Final metrics
    val_metrics = validate(model, val_loader, nn.CrossEntropyLoss(), device, num_classes)
    
    print("\n" + "="*60)
    print("FINAL RESULTS")
    print("="*60)
    print(f"Validation Loss:   {val_metrics['loss']:.4f}")
    print(f"Pixel Accuracy:    {val_metrics['pixel_accuracy']:.4f}")
    print(f"Mean IoU:          {val_metrics['miou']:.4f}")
    print(f"\nPer-class IoU:")
    for i, iou in enumerate(val_metrics['iou_per_class']):
        print(f"  Class {i}: {iou:.4f}")
    print("="*60)
    
    # Plot training history
    plot_training_history([history], ['Partial CE Loss'], metric='val_miou')
    
    # Visualize predictions
    print("\nVisualizing predictions...")
    visualize_predictions(model, val_dataset, num_samples=3)
    
    return model, history


# Run the demo
model, history = run_demo()